# Analisi Esplorativa (EDA) - Red Flag Detection

Questo notebook estende l'analisi esplorativa del dataset Kaggle `Suicide Watch`, focalizzandosi sulle criticità reali del progetto SINTON-IA (es. limite 2000 caratteri) e sulla presenza di noise testuale per preparare la fase successiva di Data Cleaning.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import re

sns.set_theme(style="whitegrid")

# Caricamento dataset
df = pd.read_csv('../data/raw/Suicide_Detection.csv')
df = df.dropna(subset=['text'])
df['text'] = df['text'].astype(str)

## 1. Analisi Lunghezze (Limite 2000 Caratteri)

In [ ]:
df['char_count'] = df['text'].apply(len)
print(df['char_count'].describe())

too_long = df[df['char_count'] > 2000]
print(f"\nRecord che superano i 2000 caratteri: {len(too_long)} ({len(too_long)/len(df)*100:.2f}%)")

plt.figure(figsize=(10, 5))
sns.histplot(data=df[df['char_count'] <= 5000], x='char_count', hue='class', bins=50, kde=True, palette='viridis')
plt.axvline(x=2000, color='red', linestyle='--', label='Limite App (2000 char)')
plt.title('Distribuzione della Lunghezza in Caratteri')
plt.xlabel('Numero di Caratteri')
plt.ylabel('Frequenza')
plt.legend()
plt.show()

## 2. Analisi Rumore Testuale (Noise)

In [ ]:
# 2.1 Presenza di URL
url_pattern = r'http[s]?://(?:[a-zA-Z]|[0-9]|[$-_@.&+]|[!*\(\),]|(?:%[0-9a-fA-F][0-9a-fA-F]))+'
df['has_url'] = df['text'].apply(lambda x: 1 if re.search(url_pattern, x) else 0)
print(f"Testi con URL: {df['has_url'].sum()} ({df['has_url'].sum()/len(df)*100:.2f}%)")

# 2.2 Formattazione Sporca (Newlines ripetute)
newline_pattern = r'\n{2,}'
df['has_newlines'] = df['text'].apply(lambda x: 1 if re.search(newline_pattern, x) else 0)
print(f"Testi con formattazione complessa (newlines multipli): {df['has_newlines'].sum()} ({df['has_newlines'].sum()/len(df)*100:.2f}%)")


In [ ]:
# 2.3 Incidenza Stop-words e Stop-patterns
# Set ridotto dimostrativo per confermare la necessità di rimozione
stopwords = set(['i', 'me', 'my', 'myself', 'we', 'our', 'you', 'your', 'he', 'him', 'she', 'her', 'it', 'they', 'them', 'the', 'a', 'an', 'and', 'but', 'if', 'or', 'because', 'as', 'to', 'from', 'in', 'on', 'with'])

def count_stopwords(text):
    words = str(text).lower().split()
    return sum(1 for w in words if w in stopwords)

df['word_count'] = df['text'].apply(lambda x: len(str(x).split()))
df['sw_count'] = df['text'].apply(count_stopwords)
df['sw_ratio'] = df.apply(lambda row: row['sw_count'] / row['word_count'] if row['word_count'] > 0 else 0, axis=1)

print(f"Rapporto medio di Stop-words nei testi: {df['sw_ratio'].mean()*100:.2f}%")
print("Questa altissima incidenza dimostra la necessità assoluta di applicare stopwords removal prima della codifica BERT.")